# Model Klasik HOG + SVM

Notebook ini hanya membangun model klasik berbasis fitur HOG dan Linear SVM. Evaluasi test set, confusion matrix, ROC, dan ringkasan metrik dipindahkan ke notebook evaluasi.

## 1. Ekstraksi Fitur HOG

Fitur HOG menangkap pola orientasi gradien dari tepi wajah dan masker. Fitur ini menjadi input untuk baseline Linear SVM.

In [ ]:

def extract_hog_features(images: np.ndarray) -> np.ndarray:
    hog = cv2.HOGDescriptor(
        (128, 128),
        (16, 16),
        (8, 8),
        (8, 8),
        9,
    )
    features = []
    for image in images:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        descriptor = hog.compute(gray).flatten()
        features.append(descriptor)
    return np.array(features, dtype=np.float32)


## 2. Training Model SVM

Dua baseline dilatih pada gambar full plain dan full enhanced. Model disimpan dalam dictionary agar evaluasi dilakukan terpusat di notebook evaluasi.

In [ ]:

import joblib
classical_experiments = [
    {
        'scenario_id': 'A_hog_svm_plain_full',
        'scenario_name': 'full_plain',
        'config': {'feature': 'HOG', 'classifier': 'LinearSVC', 'enhancement': False},
    },
    {
        'scenario_id': 'B_hog_svm_enhanced_full',
        'scenario_name': 'full_enhanced',
        'config': {'feature': 'HOG', 'classifier': 'LinearSVC', 'enhancement': True},
    },
]

classical_models = {}
classical_feature_sets = {}

for experiment in classical_experiments:
    scenario_id = experiment['scenario_id']
    scenario_name = experiment['scenario_name']

    X_train_img, y_train = load_images_for_classical(scenario_name, 'train')
    X_val_img, y_val = load_images_for_classical(scenario_name, 'validation')

    X_train = extract_hog_features(X_train_img)
    X_val = extract_hog_features(X_val_img)

    svm_model = make_pipeline(
        StandardScaler(),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=8000, random_state=SEED),
    )
    svm_model.fit(X_train, y_train)
    val_pred = svm_model.predict(X_val)
    val_accuracy = float(accuracy_score(y_val, val_pred))

    model_path = OUTPUT_DIR / f'model_{scenario_id}.joblib'
    joblib.dump(svm_model, model_path)

    classical_models[scenario_id] = svm_model
    classical_feature_sets[scenario_id] = {
        'scenario_name': scenario_name,
        'config': experiment['config'],
        'validation_accuracy': val_accuracy,
        'model_path': str(model_path),
    }

    print(f'Trained {scenario_id}')
    print(f'Validation accuracy: {val_accuracy:.4f}')
    print(f'Saved model: {model_path}')
